# Data Normalization & Input Scaling — Beginner's Masterclass
### Based on the GeeksforGeeks article, with worked examples and a real dataset

**Source:** [What Is Data Normalization? — GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/what-is-data-normalization/)

This Colab-compatible notebook introduces data scaling from first principles, calculates formulas by hand and in Python, visualizes the transformations, then applies them to a dataset.

**You will learn:**
1. Why feature scaling matters.
2. Min-Max normalization to `[0, 1]` or another range.
3. Standardization / Z-score scaling.
4. Decimal scaling.
5. Logarithmic transformation.
6. Unit-vector (L2) normalization.
7. How to apply scaling to data without test-set leakage.
8. How scaling can affect a K-Nearest Neighbours (KNN) classifier.

**Source note:** This notebook explains the article's concepts in original wording rather than reproducing the full copyrighted article. The article is linked above for reference.

## 1. What is data normalization?

A dataset contains **features** (input columns), and those columns may use different units and numeric ranges. For example, age might range from 18 to 90, cholesterol might be in the hundreds, and an income feature might be in the tens of thousands.

Feature scaling transforms numeric features so that their magnitudes are more comparable. This can matter for models that use distances (such as KNN and K-Means) and for gradient-based models (such as neural networks), where feature scales can affect optimization.

### Terms used here
- **Min-Max normalization:** rescale a feature to a chosen range, often `[0, 1]`.
- **Standardization / Z-score:** center a feature around mean 0 and scale it to standard deviation 1.
- **Decimal scaling:** divide by a power of ten.
- **Log transform:** compress large values, especially in long right-tailed nonnegative features.
- **Unit-vector normalization:** scale each sample vector to length 1.

These techniques do different things and are not interchangeable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Why scale input features?

Suppose a sample has `age = 5` and `annual_income = 20000` as differences from another sample. Euclidean distance is

\[
d(x,y)=\sqrt{\sum_j(x_j-y_j)^2}.
\]

The income difference can dominate the distance simply because its numbers are much larger—not necessarily because it is more meaningful. Scaling puts features on more comparable numeric scales.

**Often benefits:** KNN, K-Means, SVMs and gradient-based models.

**Often less important:** tree-based models, which generally split based on feature ordering rather than Euclidean distances.

Scaling is a preprocessing decision, not a guarantee that model accuracy will improve.

## 3. Min-Max normalization

The common `[0, 1]` formula is

\[
\boxed{x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}}
\]

- The feature minimum becomes 0.
- The feature maximum becomes 1.
- Values between them are rescaled proportionally.

### Worked example

For `x = [10, 20, 30, 40, 50]`, the minimum is 10 and maximum is 50. For `x = 30`:

\[
x'=\frac{30-10}{50-10}=0.5.
\]

Let's compute each value.

In [ ]:
values = np.array([10, 20, 30, 40, 50], dtype=float)
x_min, x_max = values.min(), values.max()
scaled = (values - x_min) / (x_max - x_min)
display(pd.DataFrame({"Original x": values, "x - minimum": values-x_min,
                      "maximum - minimum": x_max-x_min, "Min-Max x'": scaled}))
print("Original range:", (values.min(), values.max()))
print("Scaled range:", (scaled.min(), scaled.max()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(len(values)), values, marker="o")
axes[0].set_title("Original values")
axes[0].set_ylabel("Original units")
axes[1].plot(range(len(scaled)), scaled, marker="o")
axes[1].set_title("Min-Max scaled values")
axes[1].set_ylabel("Scaled value")
for ax in axes: ax.set_xlabel("Sample index"); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

### Scaling to a different range

To map to `[a, b]`:

\[
\boxed{x'=a+\frac{x-x_{\min}}{x_{\max}-x_{\min}}(b-a)}
\]

For `[-1, 1]`, set `a=-1` and `b=1`.

**Caution: outliers.** Min-Max uses the minimum and maximum. An extreme value can squeeze ordinary values into a small interval. A future test value outside the training range can scale below 0 or above 1.

In [ ]:
a, b = -1, 1
scaled_minus1_plus1 = a + (values-values.min())/(values.max()-values.min())*(b-a)
print("Scaled to [-1, 1]:", scaled_minus1_plus1)

with_outlier = np.array([10, 20, 30, 40, 500], dtype=float)
outlier_scaled = (with_outlier-with_outlier.min())/(with_outlier.max()-with_outlier.min())
display(pd.DataFrame({"Original with outlier": with_outlier, "Min-Max scaled": outlier_scaled}))

## 4. Standardization (Z-score scaling)

Standardization aims for mean 0 and standard deviation 1:

\[
\boxed{z=\frac{x-\mu}{\sigma}}
\]

Here, `μ` is the feature mean and `σ` is the feature **standard deviation**. The denominator is **not** the variance `σ²`.

- `z = 0`: at the mean.
- `z = 1`: one standard deviation above the mean.
- `z = -2`: two standard deviations below the mean.

NumPy's `std()` defaults to population standard deviation (`ddof=0`), which matches scikit-learn's `StandardScaler` convention.

In [ ]:
scores = np.array([10, 20, 30, 40, 50], dtype=float)
mean, std = scores.mean(), scores.std()
z_scores = (scores-mean)/std
print("Mean:", mean, " Standard deviation:", std)
print("Z-scores:", z_scores)
print("Mean after scaling:", z_scores.mean())
print("Standard deviation after scaling:", z_scores.std())
display(pd.DataFrame({"Original x": scores, "x - mean": scores-mean, "Z-score": z_scores}))

### Min-Max vs. standardization

| Property | Min-Max | Standardization |
|---|---|---|
| Formula | `(x - min)/(max - min)` | `(x - mean)/std` |
| Typical result | `[0, 1]` for fitted min/max | mean 0, standard deviation 1 on training data |
| Fixed range? | Yes for the fitted range | No |
| Outlier effects | Sensitive to extrema | Also affected by outliers |
| Common use | Distance methods, bounded input | Many scale-sensitive models |

**Technical clarification:** Standardization does not turn arbitrary data into a Gaussian distribution. It changes location and scale; it does not guarantee a bell-shaped distribution.

In [ ]:
comparison = pd.DataFrame({
    "Original": scores,
    "Min-Max": (scores-scores.min())/(scores.max()-scores.min()),
    "Standardized": (scores-scores.mean())/scores.std()
})
display(comparison)
comparison.plot(marker="o", figsize=(9,4), title="One feature under different scaling methods")
plt.xlabel("Sample index"); plt.ylabel("Transformed value"); plt.grid(alpha=.25); plt.show()

## 5. Decimal scaling

Decimal scaling moves the decimal point by dividing by a power of ten:

\[
\boxed{x'=\frac{x}{10^j}}
\]

Choose the smallest nonnegative integer `j` so that the maximum absolute scaled value is less than 1.

Example: `[12, 95, 430, -780]` has maximum absolute value 780. Dividing by 1000 (`j = 3`) makes the largest absolute result 0.78. Decimal scaling is simple, but it does not force a feature's minimum to zero or its standard deviation to one.

In [ ]:
decimal_values = np.array([12, 95, 430, -780], dtype=float)
max_abs = np.max(np.abs(decimal_values))
j = 0
while max_abs/(10**j) >= 1: j += 1
decimal_scaled = decimal_values/(10**j)
print("Original:", decimal_values, " max |x|:", max_abs)
print("j:", j, " divisor:", 10**j, " scaled:", decimal_scaled)
print("max absolute scaled value:", np.max(np.abs(decimal_scaled)))

## 6. Logarithmic transformation

The article introduces a log transform of the form

\[
\boxed{x'=\log(x+1)}
\]

A log transform compresses large values more than small values and can help with long right-tailed data.

**Domain caution:** `log(x + 1)` requires `x > -1`; for ordinary nonnegative values it is safe. It changes the distribution's shape and does **not** automatically scale all features into `[0, 1]`. Do not apply it blindly to negative-valued features. In NumPy, `np.log1p(x)` computes `log(1+x)` accurately near zero.

In [ ]:
skewed = np.array([0, 1, 2, 5, 10, 50, 100, 500, 1000], dtype=float)
logged = np.log1p(skewed)
display(pd.DataFrame({"Original": skewed, "log(1 + x)": logged}))
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].hist(skewed, bins=8); axes[0].set_title("Original values"); axes[0].set_xlabel("Value")
axes[1].hist(logged, bins=8); axes[1].set_title("After log(1 + x)"); axes[1].set_xlabel("Transformed value")
for ax in axes: ax.set_ylabel("Frequency"); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 7. Unit-vector (L2) normalization

This technique scales each **sample vector** to have length 1:

\[
\boxed{x'=\frac{x}{\lVert x\rVert_2}}
\]

For `x = [x₁, ..., xᵈ]`:

\[
\lVert x\rVert_2=\sqrt{x_1^2+x_2^2+\cdots+x_d^2}.
\]

Example: `x = [3, 4]`, so the norm is `sqrt(3² + 4²) = 5`; the unit vector is `[0.6, 0.8]`. Direction is preserved, but overall magnitude is removed. Unlike Min-Max and standardization, this is usually applied **row-wise** to each example.

In [ ]:
vector = np.array([3.0, 4.0])
unit = vector/np.linalg.norm(vector)
print("Vector:", vector, " norm:", np.linalg.norm(vector))
print("Unit vector:", unit, " norm:", np.linalg.norm(unit))
plt.figure(figsize=(6,6))
plt.axhline(0,color="gray",lw=1); plt.axvline(0,color="gray",lw=1)
plt.quiver(0,0,vector[0],vector[1],angles="xy",scale_units="xy",scale=1,label="Original [3, 4]")
plt.quiver(0,0,unit[0],unit[1],angles="xy",scale_units="xy",scale=1,label="Unit vector")
plt.xlim(-.2,5); plt.ylim(-.2,5); plt.gca().set_aspect("equal")
plt.title("L2 normalization changes length, not direction"); plt.legend(); plt.grid(alpha=.25); plt.show()

## 8. Download the heart-disease dataset used in the article

The GeeksforGeeks article demonstrates scaling numeric heart-disease features such as `age`, `trestbps`, `chol`, `thalach`, and `oldpeak`, and keeps `target` separate.

We'll attempt to download the CSV linked by the article. If that source is temporarily unavailable, the notebook falls back to scikit-learn's built-in breast-cancer dataset so that the rest of the lesson can still run. The notebook prints which dataset was loaded.

In [ ]:
from sklearn.datasets import load_breast_cancer

HEART_CSV_URL = "https://media.geeksforgeeks.org/wp-content/uploads/20250911122351383969/heart.csv"
df, dataset_name = None, None
try:
    df = pd.read_csv(HEART_CSV_URL)
    if df.empty: raise ValueError("Downloaded CSV is empty")
    dataset_name = "GeeksforGeeks heart-disease CSV"
except Exception as e:
    print("Could not download article CSV:", repr(e))
    print("Falling back to scikit-learn's built-in breast-cancer dataset.")
    fallback = load_breast_cancer(as_frame=True)
    df = fallback.frame.copy()
    dataset_name = "scikit-learn breast-cancer dataset (fallback)"

print("Dataset:", dataset_name, "| shape:", df.shape)
display(df.head())

## 9. Inspect features and target

Before scaling, inspect column names, types, missing values, and ranges. The target column represents what the model is trying to predict. Keep it separate from the input features; don't scale it as though it were a measurement.

In [ ]:
print("Columns:", df.columns.tolist())
print("Data types:"); display(df.dtypes.to_frame("dtype"))
print("Missing values:"); display(df.isna().sum().to_frame("missing count"))
print("Numeric summary:"); display(df.describe().T)

## 10. Select input features

The article uses this subset when the heart dataset is available:

```python
features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
X = df.drop('target', axis=1)
y = df['target']
```

We'll use those columns when present. For the fallback dataset, we'll choose five numeric input features. This keeps the rest of the notebook runnable regardless of which source was loaded.

In [ ]:
target_candidates = [c for c in df.columns if c.lower() in ("target", "label", "class")]
if not target_candidates:
    raise ValueError("Could not identify a target column in the loaded dataset.")
target_column = target_candidates[0]
article_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
features = [c for c in article_features if c in df.columns]
if not features:
    features = [c for c in df.select_dtypes(include=np.number).columns if c != target_column][:5]

X = df[features].copy()
y = df[target_column].copy()
valid = X.notna().all(axis=1) & y.notna()
X, y = X.loc[valid].astype(float), y.loc[valid]
print("Features:", features)
print("Target:", target_column)
print("X shape:", X.shape, " y length:", len(y))
display(X.head()); display(y.head().to_frame("target"))

## 11. Compare raw feature scales

This summary shows the minimum, maximum, mean, and standard deviation of each input. Different units and ranges are a signal to consider scaling. Plot raw boxplots to see the scale differences.

In [ ]:
before = pd.DataFrame({"min":X.min(), "max":X.max(), "mean":X.mean(), "std (population)":X.std(ddof=0)})
display(before)
plt.figure(figsize=(10,4)); X.boxplot(rot=30); plt.title("Selected features before scaling"); plt.ylabel("Original units"); plt.tight_layout(); plt.show()

## 12. Split first, then fit the scaler

For predictive evaluation, always split the dataset **before fitting** a scaler.

Correct sequence:
1. Split into training and test sets.
2. Fit the scaler using training features only.
3. Transform the training features.
4. Transform test features using the same fitted scaler.

If the scaler is fitted on the test set or the entire dataset before splitting, information about the test distribution leaks into preprocessing. That makes the evaluation less faithful to real-world prediction on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )
print("Training rows:", len(X_train), " Test rows:", len(X_test))
print("Training target distribution:"); display(y_train.value_counts().to_frame("count"))

## 13. Min-Max scale the article's input features

`MinMaxScaler` learns feature-wise minima and maxima during `fit`, and applies them during `transform`. We fit only on training data. The same fitted transformation is used on test data.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

minmax = MinMaxScaler()
X_train_minmax = pd.DataFrame(minmax.fit_transform(X_train), columns=features, index=X_train.index)
X_test_minmax = pd.DataFrame(minmax.transform(X_test), columns=features, index=X_test.index)
print("Training data after Min-Max scaling:"); display(X_train_minmax.head())
print("Training feature ranges:"); display(pd.DataFrame({"min":X_train_minmax.min(), "max":X_train_minmax.max()}))

In [ ]:
sample_idx = X_train.index[:8]
side_by_side = pd.concat([X_train.loc[sample_idx].add_suffix(" (raw)"),
                          X_train_minmax.loc[sample_idx].add_suffix(" (Min-Max)")], axis=1)
display(side_by_side)

## 14. Standardize the same features

`StandardScaler` learns the training mean and standard deviation, then transforms training and test features with those same parameters.

\[
z=\frac{x-\mu_{train}}{\sigma_{train}}
\]

The training data should have means close to zero and population standard deviations close to one (constant columns are a special case).

In [ ]:
standard = StandardScaler()
X_train_std = pd.DataFrame(standard.fit_transform(X_train), columns=features, index=X_train.index)
X_test_std = pd.DataFrame(standard.transform(X_test), columns=features, index=X_test.index)
display(pd.DataFrame({"mean after scaling":X_train_std.mean(),
                      "population std after scaling":X_train_std.std(ddof=0)}).round(5))

## 15. Visualize one real feature under three representations

The basic distribution shape generally remains similar after linear scaling, but its location and scale change.

In [ ]:
feature_to_plot = features[0]
fig, axes = plt.subplots(1,3,figsize=(14,4))
axes[0].hist(X_train[feature_to_plot], bins=20); axes[0].set_title(f"Raw: {feature_to_plot}"); axes[0].set_xlabel("Original units")
axes[1].hist(X_train_minmax[feature_to_plot], bins=20); axes[1].set_title("Min-Max"); axes[1].set_xlabel("Scaled value")
axes[2].hist(X_train_std[feature_to_plot], bins=20); axes[2].set_title("Standardized"); axes[2].set_xlabel("Z-score")
for ax in axes: ax.set_ylabel("Frequency"); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 16. Decimal scaling on the selected features

Choose the smallest integer `j` separately for each feature, using the training maximum absolute value. Then apply the same divisors to the test set.

In [ ]:
def fit_decimal_exponents(frame):
    js = {}
    for col in frame.columns:
        max_abs = float(frame[col].abs().max())
        j = 0
        while max_abs/(10**j) >= 1: j += 1
        js[col] = j
    return pd.Series(js, dtype=int)

decimal_j = fit_decimal_exponents(X_train)
decimal_divisors = pd.Series(10.0 ** decimal_j, index=decimal_j.index)
X_train_decimal = X_train / decimal_divisors
X_test_decimal = X_test / decimal_divisors
print("Exponent j per feature:"); display(decimal_j.to_frame("j"))
print("Max absolute values after decimal scaling:"); display(X_train_decimal.abs().max().to_frame("max |scaled x|"))

## 17. Log transform appropriate nonnegative features

The article's formula is `log(x + 1)`. We'll apply this only where all train and test values of a selected feature are nonnegative. `np.log1p` calculates `log(1+x)` accurately for values close to zero.

In [ ]:
log_features = [c for c in features if (X_train[c] >= 0).all() and (X_test[c] >= 0).all()]
X_train_log, X_test_log = X_train.copy(), X_test.copy()
if log_features:
    X_train_log[log_features] = np.log1p(X_train[log_features])
    X_test_log[log_features] = np.log1p(X_test[log_features])
    print("Log-transformed features:", log_features)
    display(X_train_log[log_features].head())
else:
    print("No selected feature is nonnegative in both train and test; no log transform applied.")

## 18. Unit-vector normalization on dataset rows

The `Normalizer` applies L2 normalization row by row. Each example vector is divided by its own L2 norm; this is different from scaling each column independently.

In [ ]:
normalizer = Normalizer(norm="l2")
X_train_unit = pd.DataFrame(normalizer.fit_transform(X_train), columns=features, index=X_train.index)
X_test_unit = pd.DataFrame(normalizer.transform(X_test), columns=features, index=X_test.index)
print("First five row norms:", np.linalg.norm(X_train_unit.to_numpy()[:5], axis=1))
display(X_train_unit.head())

## 19. Does scaling change KNN performance?

KNN uses distances, so feature scale can matter. We'll compare raw features, Min-Max scaling, and standardization with the same train/test split and `k=5`.

The scalers are placed inside scikit-learn pipelines so they are fitted on training data only. The exact accuracy can change with the dataset; one scaler is not always best.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_models = {
    "Raw features": make_pipeline(SimpleImputer(strategy="median"), KNeighborsClassifier(n_neighbors=5)),
    "Min-Max scaling": make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Standardization": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
results=[]
for name, model in knn_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"Preprocessing":name, "Test accuracy (%)":round(accuracy_score(y_test,pred)*100,2)})
results_df = pd.DataFrame(results).sort_values("Test accuracy (%)", ascending=False)
display(results_df)

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(results_df["Preprocessing"], results_df["Test accuracy (%)"])
plt.ylim(0,100); plt.ylabel("Test accuracy (%)"); plt.title("KNN performance under different preprocessing choices")
plt.xticks(rotation=15); plt.grid(axis="y", alpha=.25); plt.tight_layout(); plt.show()

## 20. Which technique should you choose?

| Technique | What it does | Common use | Main caution |
|---|---|---|---|
| Min-Max | Maps training min/max to a target range | Distance methods, bounded inputs | Sensitive to outliers; future values can leave the range |
| Standardization | Training mean 0, standard deviation 1 | Many scale-sensitive models | Does not make data Gaussian; outliers can affect statistics |
| Decimal scaling | Divides by a power of 10 | Simple magnitude reduction | Doesn't ensure a shared fixed range |
| Log transform | Compresses large values | Right-skewed nonnegative variables | Must stay within the logarithm's domain |
| Unit-vector normalization | Each row has length 1 | Direction/profile-based comparison | Removes total magnitude information |

### Practical rules of thumb
- **KNN / K-Means:** scaling is usually important.
- **Neural networks:** scaled inputs often make optimization easier; choose based on your data and architecture.
- **Long-tailed nonnegative values:** consider a log transform.
- **Vectors where direction matters more than magnitude:** consider unit-vector normalization.
- **Tree-based models:** scaling is often less critical.

## 21. Common mistakes

1. Fitting the scaler before the train/test split: this can leak test-set information into preprocessing.
2. Fitting a separate scaler on the test set: use the training-fitted scaler instead.
3. Scaling the target accidentally: keep it separate unless target transformation is intentional.
4. Dividing by variance instead of standard deviation in Z-score scaling.
5. Assuming Min-Max test values must stay between 0 and 1.
6. Applying a logarithm to values outside its valid domain.
7. Thinking all methods called “normalization” perform the same transformation.
8. Assuming scaling guarantees a better score: evaluate it on held-out data.

## 22. Practice questions

Try these before running the answer cell.

1. Min-Max scale `x = 25` when minimum is 5 and maximum is 45.
2. A feature has mean 50 and standard deviation 10. Find the Z-score for `x = 70`.
3. Normalize the vector `[5, 12]` using L2 normalization.
4. Why should the scaler be fitted using training data only?
5. Which transformation might help a long right-tailed, nonnegative feature?
6. Does standardization guarantee Gaussian-distributed data?

In [ ]:
# Answer key — try the questions before running this cell.
print("Q1:", (25-5)/(45-5))
print("Q2:", (70-50)/10)
v = np.array([5.0,12.0])
print("Q3:", v/np.linalg.norm(v), " norm:", np.linalg.norm(v/np.linalg.norm(v)))
print("Q4: To avoid leaking test-set information into preprocessing.")
print("Q5: A log transform may help when all values are in its valid domain.")
print("Q6: No. Standardization changes location and scale, not the distribution's shape.")

## 23. Final recap

\[
\text{Min-Max: }x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
\]

\[
\text{Standardization: }z=\frac{x-\mu}{\sigma}
\]

\[
\text{Decimal scaling: }x'=\frac{x}{10^j}
\]

\[
\text{Log transform: }x'=\log(1+x)
\]

\[
\text{Unit vector: }x'=\frac{x}{\lVert x\rVert_2}
\]

The GeeksforGeeks article introduces these techniques and compares Min-Max normalization with standardization. This notebook adds worked calculations, visual demonstrations, dataset preprocessing, and a held-out KNN comparison.

**Reference:** [GeeksforGeeks — What Is Data Normalization?](https://www.geeksforgeeks.org/machine-learning/what-is-data-normalization/)